In [ ]:
import pandas as pd
from jiwer import wer, cer
from utils.num_to_words import numbers_to_words
import re
import evaluate

## Setup

In [7]:
def clean_text(text):
    text = re.sub(r"-", " ", text)
    text = re.sub(r"[^\w\s]", "", text)
    text = text.lower()
    return text

comet = evaluate.load('comet')

Fetching 5 files: 100%|██████████| 5/5 [02:06<00:00, 25.26s/it]
I0615 09:53:46.471071 6180 utils.py:154] Lightning automatically upgraded your loaded checkpoint from v1.8.3.post1 to v2.5.1.post0. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint C:\Users\juliu\.cache\huggingface\hub\models--Unbabel--wmt22-comet-da\snapshots\2760a223ac957f30acfb18c8aa649b01cf1d75f2\checkpoints\model.ckpt`
W0615 09:53:47.352595 6180 warnings.py:112] c:\Users\juliu\Miniconda3\envs\02466_AI_dubbing\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\juliu\.cache\huggingface\hub\models--xlm-roberta-large. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environme

In [9]:
speakers = [
    "speaker_1", "speaker_2", "speaker_3", "speaker_4", "speaker_5",
    "speaker_6", "speaker_7", "speaker_8", "speaker_9", "speaker_10"
]

chunk_sizes = [3000, 3500, 4000, 4500, 5000, 5500, 6000]

## Data preprocessing 

In [ ]:
df = pd.DataFrame(columns=['language','chunk_size', 'speaker', 'wer', 'cer', 'comet-score', 'stt_time', 'tt_time', 'tts_time', 'total_time'])

# Danish
for size in chunk_sizes:
    for speaker in speakers:
        # Loading data
        csv_df = pd.read_csv(f'logs/dk_{speaker}_chunk{size}.csv')

        model_transcription = ' '.join(csv_df['transcription'].astype(str))
        model_translation = ' '.join(csv_df['translation'].astype(str))

        with open(f"data/danish/dk_{speaker}_final.txt", "r", encoding="utf-8") as f:
            ref_transcription = f.read()
        with open(f"data/english/{speaker}_final.txt", "r", encoding="utf-8") as f:
            ref_translation = f.read()

        # Cleaning data
        model_transcription = clean_text(model_transcription)
        model_transcription = numbers_to_words(model_transcription, 'da', split_abbreviations=False)
        model_transcription = model_transcription.replace(' øøhhm', '')

        model_translation = clean_text(model_translation)
        model_translation = numbers_to_words(model_translation, 'en', split_abbreviations=False)
        model_translation = model_translation.replace(' ohhm', '')

        ref_transcription = clean_text(ref_transcription)
        ref_transcription = numbers_to_words(ref_transcription, 'da')

        ref_translation = clean_text(ref_translation)
        ref_translation = numbers_to_words(ref_transcription)

        # Computing metrics
        wer_score = wer(reference=ref_transcription, hypothesis=model_transcription)
        cer_score = cer(reference=ref_transcription, hypothesis=model_transcription)

        comet_score = comet.compute(predictions=[model_translation], references=[ref_transcription], sources=[model_transcription])['score'][0]

        stt_times = csv_df['stt_latency_ms'].tolist()
        tt_times = csv_df['tt_latency_ms'].tolist()
        tts_times = csv_df['tts_latency_ms'].tolist()
        total_times = csv_df['total_latency_ms'].tolist()

        df.loc[len(df)] = ['da', size, speaker, wer_score, comet_score, stt_times, tt_times, tts_times, total_times]

# English
for size in chunk_sizes:
    for speaker in speakers:
        # Loading data
        csv_df = pd.read_csv(f'logs/{speaker}_final_chunk{size}.csv')

        model_transcription = ' '.join(csv_df['transcription'].astype(str))
        model_translation = ' '.join(csv_df['translation'].astype(str))

        with open(f"data/english/{speaker}_final.txt", "r", encoding="utf-8") as f:
            ref_transcription = f.read()
        with open(f"data/danish/dk_{speaker}_final.txt", "r", encoding="utf-8") as f:
            ref_translation = f.read()

        # Cleaning data
        model_transcription = clean_text(model_transcription)
        model_transcription = numbers_to_words(model_transcription, 'en', split_abbreviations=False)

        model_translation = clean_text(model_translation)
        model_translation = numbers_to_words(model_translation, 'da', split_abbreviations=False)

        ref_transcription = clean_text(ref_transcription)
        ref_transcription = numbers_to_words(ref_transcription, 'en', split_abbreviations=False)

        ref_translation = clean_text(ref_translation)
        ref_translation = numbers_to_words(ref_transcription, 'da', split_abbreviations=False)

        # Computing metrics
        print('ref_transcription: ', ref_transcription)
        print('hypothesis_transcription: ', model_transcription)
        wer_score = wer(reference=ref_transcription, hypothesis=model_transcription)
        cer_score = cer(reference=ref_transcription, hypothesis=model_transcription)

        comet_score = comet.compute(predictions=[model_translation], references=[ref_transcription], sources=[model_transcription])['score'][0]

        stt_times = csv_df['stt_latency_ms'].tolist()
        tt_times = csv_df['tt_latency_ms'].tolist()
        tts_times = csv_df['tts_latency_ms'].tolist()
        total_times = csv_df['total_latency_ms'].tolist()

        df.loc[len(df)] = ['en', size, speaker, wer_score, cer_score, comet_score, stt_times, tt_times, tts_times, total_times]

df.to_csv('results.csv')

I0615 10:17:32.277429 6180 callback_connector.py:108] Using default `ModelCheckpoint`. Consider installing `litmodels` package to enable `LitModelCheckpoint` for automatic upload to the Lightning model registry.


ref_transcription:  id like to share with you a discovery that i made a few months ago while writing an article for italian wired i always keep my thesaurus handy whenever im writing anything but id already finished editing the piece and i realized that i had never once in my life looked up the word disabled to see what id find let me read you the entry disabled adjective crippled helpless useless wrecked stalled maimed wounded mangled lame mutilated rundown worn out weakened impotent castrated paralyzed handicapped senile decrepit laid up done up done for done in cracked up counted out see also hurt useless and weak antonyms healthy strong capable i was reading this list out loud to a friend and at first was laughing it was so ludicrous but i just gotten past mangled and my voice broke and i had to stop and collect myself from the emotional shock uh and and impact that the assault from these words unleashed you know hum of course this is my raggedy old thesaurus im thinking this must 

I0615 10:17:32.508389 6180 setup.py:156] GPU available: False, used: False
I0615 10:17:32.509393 6180 setup.py:159] TPU available: False, using: 0 TPU cores
I0615 10:17:32.510403 6180 setup.py:169] HPU available: False, using: 0 HPUs


ref_transcription:  im going to talk today about energy and climate and that might seem a bit surprising because my full time work at the foundation is mostly about vaccines and seeds about the things that we need to invent and deliver to help the poorest two billion live better lives but energy and climate are extremely important to these people in fact more important than to anyone else on the planet the climate getting worse means that many years their crops wont grow there will be too much rain not enough rain things will change in ways that their fragile environment simply cant support and that leads to starvation it leads to uncertainty it leads to unrest so the the climate changes will be terrible for them also the price of energy is very important to them in fact if you could pick just one thing to lower the price of to reduce poverty by far you would pick energy now the price of energy has come down over time uh really advanced civilization is based on advances in energy the c

I0615 10:17:52.186209 6180 callback_connector.py:108] Using default `ModelCheckpoint`. Consider installing `litmodels` package to enable `LitModelCheckpoint` for automatic upload to the Lightning model registry.
I0615 10:17:52.305049 6180 setup.py:156] GPU available: False, used: False
I0615 10:17:52.306050 6180 setup.py:159] TPU available: False, using: 0 TPU cores
I0615 10:17:52.312242 6180 setup.py:169] HPU available: False, using: 0 HPUs
I0615 10:17:56.868712 6180 callback_connector.py:108] Using default `ModelCheckpoint`. Consider installing `litmodels` package to enable `LitModelCheckpoint` for automatic upload to the Lightning model registry.
I0615 10:17:56.959123 6180 setup.py:156] GPU available: False, used: False
I0615 10:17:56.960157 6180 setup.py:159] TPU available: False, using: 0 TPU cores
I0615 10:17:56.960157 6180 setup.py:169] HPU available: False, using: 0 HPUs


...

ref_transcription:  if i can leave you with one big idea today its that the whole of the data in which we consume is greater that the sum of the parts and instead of thinking about information overload what id like you to think about is how we can use information so that patterns pop and we can see trends that would otherwise be invisible so what were looking at right here is a typical mortality chart organized by age this tool that im using here is a little experiment its called pivot and with pivot what i can do is i can choose to filter in one particular cause of deaths say accidents and right away i see theres a different pattern that emerges this is because in the mid area here people are at their most active and over here theyre at their most frail we can step back out again and then reorganize the data by cause of death seeing that circulatory diseases and cancer are the usual suspects but not for everyone if we go ahead and we filter by age say forty years or less we see that a

I0615 10:18:23.010751 6180 setup.py:156] GPU available: False, used: False
I0615 10:18:23.014766 6180 setup.py:159] TPU available: False, using: 0 TPU cores
I0615 10:18:23.016759 6180 setup.py:169] HPU available: False, using: 0 HPUs
I0615 10:18:33.135326 6180 callback_connector.py:108] Using default `ModelCheckpoint`. Consider installing `litmodels` package to enable `LitModelCheckpoint` for automatic upload to the Lightning model registry.


ref_transcription:  i grew up on a steady diet of science fiction in high school i i took a bus to school an hour each way every day and i was always absorbed in a book science fiction book which took my mind to other worlds and satisfied this in in a in a narrative form this insatiable sense of curiosity that i had and and you know that curiosity also manifested itself in in the fact that whenever i wasnt in school i was i was out in the woods hiking and taking samples frogs and snakes and bugs and pond water and bringing it back looking at it under the microscope i was you know i was a real science geek but it was all about trying to understand understand the world understand the the limits of of possibility and my you know love of of science fiction actually seemed to be mirrored in the world around me because what was happening this is in the late sixties you know we were we were going to the moon we were exploring the deep oceans jacques cousteau was coming into our living rooms w

I0615 10:18:33.339313 6180 setup.py:156] GPU available: False, used: False
I0615 10:18:33.340376 6180 setup.py:159] TPU available: False, using: 0 TPU cores
I0615 10:18:33.341353 6180 setup.py:169] HPU available: False, using: 0 HPUs
I0615 10:18:35.302298 6180 call.py:57] 
Detected KeyboardInterrupt, attempting graceful shutdown ...


NameError: name 'exit' is not defined

## data vizualisation

In [16]:
df

,language,chunk_size,speaker,wer,comet-score,stt_time,tt_time,tts_time,total_time
0,en,3000,speaker_1,0.181435,"{'mean_score': 0.5233229398727417, 'scores': [...","[554.3, 611.49, 509.93, 639.52, 624.08, 572.07...","[694.31, 1089.04, 1080.62, 1317.73, 1373.31, 1...","[7205.89, 8885.2, 3010.32, 6003.2, 3927.24, 39...","[10530.53, 15739.96, 15688.94, 18623.27, 19284..."
1,en,3000,speaker_2,0.131034,"{'mean_score': 0.6031394004821777, 'scores': [...","[544.36, 561.61, 555.64, 498.61, 524.69, 497.5...","[476.26, 784.63, 900.63, 881.02, 923.07, 884.4...","[4054.19, 6054.14, 4610.48, 2615.34, 2978.01, ...","[8270.88, 11264.67, 12605.6, 12159.73, 12072.3..."
2,en,3000,speaker_3,0.198083,"{'mean_score': 0.5507157444953918, 'scores': [...","[558.4, 519.07, 514.12, 538.12, 490.56, 497.08...","[727.62, 772.61, 678.11, 1033.21, 1126.5, 895....","[6386.0, 1804.25, 1916.13, 5164.65, 2530.59, 2...","[11092.16, 9421.43, 8268.52, 9959.53, 9233.58,..."
3,en,3000,speaker_4,0.190332,"{'mean_score': 0.5584998726844788, 'scores': [...","[521.41, 550.56, 542.26, 572.91, 543.32, 593.7...","[547.11, 921.2, 823.63, 913.26, 827.54, 722.55...","[5461.21, 6097.7, 2769.59, 3392.59, 2698.77, 2...","[9617.9, 12660.97, 12365.58, 12688.38, 12323.2..."
4,en,3000,speaker_5,0.104000,"{'mean_score': 0.6144610047340393, 'scores': [...","[548.61, 510.72, 578.07, 534.65, 468.39, 598.2...","[550.06, 896.12, 1090.94, 1106.13, 798.83, 878...","[7222.31, 4760.31, 6324.65, 2808.94, 2471.99, ...","[11826.78, 13477.54, 16684.77, 16161.79, 15525..."
5,en,3000,speaker_6,0.177215,"{'mean_score': 0.6460681557655334, 'scores': [...","[587.66, 575.38, 543.36, 536.82, 544.32, 530.2...","[808.22, 1083.61, 1016.24, 938.49, 910.31, 100...","[9958.56, 3517.95, 2672.52, 2299.8, 3150.01, 5...","[14221.6, 14605.98, 13946.54, 13132.02, 12955...."
